# Phase 5: SFT (+ optional DPO), evaluation, Hub export, browser demo
Accelerator: **GPU T4 x2**. SFT takes ~1.5 h; DPO ~1 h.

In [ ]:
# Kaggle: Add-ons -> Secrets: HF_TOKEN (write), HF_USERNAME, WANDB_API_KEY. Settings -> Internet ON.
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ["HF_TOKEN", "HF_USERNAME", "WANDB_API_KEY"]:
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        print(f"secret {name} not set")

GITHUB_REPO = "https://github.com/Dhagash1206/MiniLLM.git"
%cd /kaggle/working
!git clone -q {GITHUB_REPO} minillm 2>/dev/null || (cd minillm && git pull -q)
%cd /kaggle/working/minillm
!pip install -q -e .

## SFT (add `sft.lora_enabled=true` for LoRA)

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m MiniLLM.Training.SFT --config Configs/Run125M.yaml

## Optional: DPO on UltraFeedback

In [ ]:
RUN_DPO = False
if RUN_DPO:
    !torchrun --standalone --nproc_per_node=2 -m MiniLLM.Training.DPO --config Configs/Run125M.yaml

## Evaluate

In [ ]:
FINAL = "checkpoints/dpo/latest" if RUN_DPO else "checkpoints/sft/latest"
TOK = "data/tokens/tokenizer"
!python -m MiniLLM.Evaluation.HellaSwag --model {FINAL} --tokenizer {TOK}
!python -m MiniLLM.Evaluation.MBPP --model {FINAL} --tokenizer {TOK} --chat --output results/mbpp_chat.json
!python -m MiniLLM.Evaluation.PromptReport --model {FINAL} --tokenizer {TOK} --chat --output results/prompts.md

## Export to the Hub (Llama + ONNX) and deploy the static chat Space

In [ ]:
# The chat repo is public so visitors' browsers can download the ONNX files.
!python -m MiniLLM.HFExport --checkpoint {FINAL} --tokenizer {TOK} --out artifacts/hf_model \
    --push-to-hub $HF_USERNAME/minillm-125m-chat
!pip install -q -e ".[web]"
!python -m MiniLLM.OnnxExport --model artifacts/hf_model --push-to-hub $HF_USERNAME/minillm-125m-chat
!python Scripts/DeploySpace.py --space $HF_USERNAME/minillm-chat --model-repo $HF_USERNAME/minillm-125m-chat